# 01 — Regimes and Markov Chains

*The founding notebook: from "markets feel like they have moods" to a full toolkit —
simulation, stationary distributions, first passage, spells, matrix powers, and the
Bonus-Malus premium machine. Covers Ross 4.1–4.3 (and previews 4.4), arrived at backwards.*

**How to read this notebook:** every claim runs. Prose cells marked ✍️ are for MY OWN words
(retrieval practice — writing them IS the revision). Code cells are the witnesses.

---
## 1. The problem before the maths

✍️ *The practical problem that sparked this is how to model shifting price regimes for a commodity such as bunker fuel prices. If the regimes are discrete, what can we tell about the transition between the regimes or states. Can we describe those movements over time? E.g. What fraction of time is spent volatile? How long does it usually take to return to quiet? I also just want to understand prob models and markov chains. So there can be a synergy here. The problem can lead or the maths can lead. But as regards the learning, want to do it through modelling first and reading second.*



In [1]:
import numpy as np

from labs.prob_models import bonus_malus as bm
from labs.prob_models import regime_chain as rc

# Functions stay namespaced, so it is always obvious which model a call belongs
# to (both modules export a `simulate`). The chain under study is bound locally
# because it is the subject of the notebook, not a detail of some module.
T, LABELS = rc.T, rc.LABELS

print(LABELS)
print(T)

['Quiet', 'Trending', 'Volatile']
[[0.9  0.08 0.02]
 [0.2  0.7  0.1 ]
 [0.1  0.3  0.6 ]]


## 2. The naive build: loops and rules

✍️ *We assume three states: Quiet, Trending and Volatile and do a 3x3 transition matrix between the three.  Note that each row sums to 1 as the state must move somewhere. Diagonals dominate as states are somewhat sticky. These parameters should come from examining empirical data/trade knowledge. The value is this approach is that that a global texture can be created from that local knowledge. Note that the 3 bins and their definitions are assumed. Something to look into later. We simulate a market time series from the transition matrix first. To calculate storm length we build a list of storm lengths ie first entry into storm state until calm state. Trending days mid storm count. For a more nuanced question like volatile days per excursion from quiet, we must hold an in_spell flag as as we need to track that state as well as the volatile days counters*

**Predictions I made before ever running it** (keep these — the gaps were the curriculum):
- Fraction of time volatile: guessed **7%** → truth **9.9%**
- Days from volatile back to quiet: guessed **5** → truth **6.67**

In [2]:
rng = np.random.default_rng(42)
hist = rc.simulate(T, 200_000, rng=rng)
pi_sim = rc.empirical_distribution(hist, 3)
print('time-shares (simulated):', dict(zip(LABELS, pi_sim.round(4))))

time-shares (simulated): {'Quiet': np.float64(0.6309), 'Trending': np.float64(0.269), 'Volatile': np.float64(0.1002)}


## 3. Is there maths for this? — 4.1 and the Markov property

✍️ *TODO: equation 4.1 in my words. The pipe = given. The history appears on the left
and VANISHES on the right — that's the assumption, not a theorem: the state is a
sufficient statistic, the past is banked. Also: the dimensionality cheat — nothing is
Markov in the wrong state space, nearly everything is in a rich enough one (weather
example: two-day states). Memory is relocated, not removed; paid for in dimensions.*

**Ross:** §4.1, eq (4.1); Example 4.4 (transforming a process into a chain).

## 4. Two questions, two tools

### 4a. Spell length — geometric, memoryless
✍️ *TODO: pure-volatility runs. Each day: stay 0.6 / leave 0.4 ⇒ geometric, mean 1/0.4 = 2.5.
Why no system of equations needed. THE BUGS: sentinel d=-1 made one variable do two jobs
(am-I-in-a-spell boolean + counter); single-day spells silently dropped; the buggy version
was accidentally unbiased ONLY because geometric. Flag and counter are separate concepts.*

### 4b. Storm length — first passage
✍️ *TODO: volatile until QUIET, trending detours count. First-step analysis:
h_V = 1 + 0.6 h_V + 0.3 h_T; h_T = 1 + 0.7 h_T + 0.1 h_V ⇒ h_V = 20/3.
Why 6.67 ≫ 2.5: escape usually lands in Trending, a bad neighbourhood that routes back.
The storm is brief; the unsettled weather is long. (Commercially: 6.67 is the number
for "when to fix a price after a shock", not 2.5 — but 2.5 is what naive eyeballing reports.)*

In [3]:
spells = rc.spell_lengths(hist, state=2)
storms = rc.storm_lengths(hist)
h = rc.mean_first_passage(T, target=0)
print(f'mean volatile spell: {np.mean(spells):.3f}   (theory 2.5)')
print(f'mean storm V→Q:      {np.mean(storms):.3f}   (theory 20/3 = {20/3:.3f})')
print(f'first passage h:     {h.round(3)}  (Q, T, V)')

mean volatile spell: 2.493   (theory 2.5)
mean storm V→Q:      6.733   (theory 20/3 = 6.667)
first passage h:     [0.    5.556 6.667]  (Q, T, V)


## 5. Chapman–Kolmogorov: splicing journeys

✍️ *TODO: derived at n=m=1 by hand (P²₀₂ = via-Q + via-T + via-V). The four moves:
partition by waypoint (sort, don't compute) → chain rule (story order) → MARKOV deletes
the origin behind the bar (the one real step) → rename. Σₖ Pⁿᵢₖ Pᵐₖⱼ is row-times-column
⇒ n-step matrix IS the n-th matrix power. Scalar trap: P¹⁰ₐᵦ ≠ (P⁵ₐᵦ)² — matrices multiply,
entries don't. Powers of the same matrix commute = slice the journey anywhere.*

**Ross:** §4.2, eq (4.2). One-liner for the lore: *splice journeys at a checkpoint;
matrix multiplication does the splicing.*

In [4]:
# CK witnessed three ways
P2_hand = sum(T[0, k] * T[k, 2] for k in range(3))
print(f'P²(Q→V) by hand-sum: {P2_hand:.4f}')
print(f'P²(Q→V) by matrix:   {np.linalg.matrix_power(T, 2)[0, 2]:.4f}')
P10 = np.linalg.matrix_power(T, 10)
P5 = np.linalg.matrix_power(T, 5)
print('P¹⁰ == P⁵ @ P⁵ :', np.allclose(P10, P5 @ P5))
print(f'scalar trap: (P⁵₂₀)² = {P5[2,0]**2:.4f}  vs  P¹⁰₂₀ = {P10[2,0]:.4f}  (different!)')

P²(Q→V) by hand-sum: 0.0380
P²(Q→V) by matrix:   0.0380
P¹⁰ == P⁵ @ P⁵ : True
scalar trap: (P⁵₂₀)² = 0.2023  vs  P¹⁰₂₀ = 0.5935  (different!)


## 6. The origin washes out: π three ways

✍️ *TODO: rows of Pⁿ flatten to identical rows — a matrix that has forgotten its row index.
π = long-run TIME-SHARES (occupancy, not residence; nothing "ends up" anywhere).
My wrong guess: column-sums normalised = one iteration of the power method starting uniform.
The fix-point: πP = π needs the sum-to-1 constraint; the all-ones-matrix trick folds it in.*

In [5]:
for n in [1, 2, 5, 20, 50]:
    print(f'P^{n:2d} row Q:', np.linalg.matrix_power(T, n)[0].round(4))
print('\npi by power :', rc.stationary_by_power(T).round(4))
print('pi by solve :', rc.stationary_by_solve(T).round(4))
print('pi by sim   :', pi_sim.round(4))

P^ 1 row Q: [0.9  0.08 0.02]
P^ 2 row Q: [0.828 0.134 0.038]
P^ 5 row Q: [0.71   0.2168 0.0731]
P^20 row Q: [0.6345 0.2671 0.0983]
P^50 row Q: [0.6338 0.2676 0.0986]

pi by power : [0.6338 0.2676 0.0986]
pi by solve : [0.6338 0.2676 0.0986]
pi by sim   : [0.6309 0.269  0.1002]


## 7. Classification: does the chain come back? (Ross 4.3)

✍️ *TODO in my words:*
- *Accessible (one-way) vs communicate (two-way, ↔, dialogue not press release).
  Classes = islands of mutual reach; between islands all arrows one-way (else they merge).*
- *RECURRENT = certain RETURN (not "stay"! absorbing is the corner case). Bootstrap:
  certain return, fresh start, certain again ⇒ no last visit ⇒ infinite visits.*
- *TRANSIENT = each visit risks being the last (prob 1−f) ⇒ visit count geometric,
  mean 1/(1−f) — same machine as the volatile spell, coin relabelled.*
- *The census test (Prop 4.1): Σₙ Pⁿᵢᵢ = expected visits (couch cat: photos count
  APPEARANCES; the union asks about DAYS — hence ≤). Infinite census ⟺ recurrent.*
- *Contagion: same class, same fate — detour through a recurrent classmate, CK splices,
  infinity survives the a·b toll. Finite one-class chains ⇒ all recurrent (the warranty
  card for π existing at all).*
- *Drunk walk: P²ⁿ₀₀ ~ 1/√(πn); Σ diverges ⇒ recurrent in 1D (and 2D, barely);
  converges in 3D ⇒ the drunk bird is lost. Certain return + INFINITE expected return
  time is possible (St Petersburg / fat tails: the mean is a census rare giants bankrupt).*

In [6]:
# Example 4.15's four-state chain: classes {0,1}, {2}, {3}
P415 = np.array([[.5,.5,0,0],[.5,.5,0,0],[.25,.25,.25,.25],[0,0,0,1]])
# census for state 2 (transient): Σₙ₌₀ Pⁿ₂₂ = expected TOTAL visits (n=0 term = the initial visit)
census = sum(np.linalg.matrix_power(P415, n)[2, 2] for n in range(200))
print(f'state 2 census Σ Pⁿ₂₂ ≈ {census:.4f}  (finite ⇒ transient)')
print('theory: f₂ = ¼ (only return route is staying put)',
      f'⇒ visits geometric, mean 1/(1-¼) = {1/0.75:.4f} ✓')
# fair-walk return probs P^{2n}₀₀ = C(2n,n)/4ⁿ — computed by ratio to dodge overflow
ps, p = [], 1.0
for n in range(1, 200_000):
    p *= (2*n - 1) / (2*n)          # C(2n,n)/4^n = prev * (2n-1)/(2n)
    ps.append(p)
cs = np.cumsum(ps)
print(f'walk census partial sums: n=100: {cs[99]:.2f}',
      f'  n=10k: {cs[9999]:.2f}   n=200k: {cs[-1]:.2f}',
      '\n→ still growing like 2√(n/π), no ceiling ⇒ diverges ⇒ RECURRENT (drunk man gets home)')

state 2 census Σ Pⁿ₂₂ ≈ 1.3333  (finite ⇒ transient)
theory: f₂ = ¼ (only return route is staying put) ⇒ visits geometric, mean 1/(1-¼) = 1.3333 ✓
walk census partial sums: n=100: 10.33   n=10k: 111.84   n=200k: 503.63 
→ still growing like 2√(n/π), no ceiling ⇒ diverges ⇒ RECURRENT (drunk man gets home)


## 8. The premium machine: Bonus-Malus end-to-end

✍️ *TODO: the nature/posited split — Poisson(λ) is NATURE (estimated, falsifiable);
the table s(i,k) is POSITED (designed, auditable). P is MANUFACTURED:
"sum the likelihoods of all the k's that bring you from i to j" — k is a dummy,
the colon means such-that, the Σ only earns its keep where counts pool (last column,
= 1 − head by the complement trick). Rows-sum-to-1 is the checksum for a mislaid k.
Also: interior states can't self-loop (designed anti-stickiness) — occupancy without residence.*

In [7]:
lam = 1.0
P = bm.build_matrix(lam)
pi = bm.stationary(P)
print('P:')
print(P.round(4))
print('pi:', pi.round(4))
print(f'average premium: {pi @ bm.PREMIUMS:.2f}')
print('\n✍️ TODO note: with λ=1, HALF of policyholders live in state 4 long-run —',
      '\nthe ladder is slow-up fast-down BY DESIGN. What happens to π and the premium as λ → 0.3?')

P:
[[0.3679 0.3679 0.1839 0.0803]
 [0.3679 0.     0.3679 0.2642]
 [0.     0.3679 0.     0.6321]
 [0.     0.     0.3679 0.6321]]
pi: [0.0707 0.1214 0.2594 0.5484]
average premium: 477.34

✍️ TODO note: with λ=1, HALF of policyholders live in state 4 long-run — 
the ladder is slow-up fast-down BY DESIGN. What happens to π and the premium as λ → 0.3?


---
## 9. Exercises (LLM-free zone)

**Protocol:** attempt cold at the START of the next session (cold-start = revision +
gap-detector + re-entry all at once). Genuine attempt before any post-mortem.

**From Ross Ch. 4 problem set** *(TODO: pick and pencil in problem numbers from my edition)*:
- [ ] One **drill** from the first ~10 (build a matrix / compute a 2-step probability)
- [ ] Two **modelling** problems from the middle band — word problem, CHOOSE THE STATE
      SPACE MYSELF (the transferable skill; expect a dimensionality-cheat moment)
- [ ] One **classification** problem (identify classes, transient/recurrent, justify via census or structure)

**Own exercises (predict → compute → break):**
- [ ] *Predict:* for the regime chain with row V changed to [0.05, 0.15, 0.80] — which way
      do π_V and h_V move, and which moves MORE? Then compute.
- [ ] *Compute (by hand):* P³(T→Q) via one CK splice of P¹ and P². Check in numpy.
- [ ] *Break:* make state 4 of Bonus-Malus absorbing (licence revoked). What is π now?
      What QUESTION replaces "long-run share"? (Hint: ending up vs time spent.)

## 10. Close-out
✍️ *TODO: one lore line for this notebook; revisit-in-2-weeks questions (3, drawn from above);
date; state of play for notebook 02.*